# Incremental Updates: Rolling Operations Forecast
# 增量更新：滚动运营预测

Scenario: a call-center or operations dashboard receives new daily observations and refreshes forecasts without rebuilding the whole workflow manually.

场景：呼叫中心或运营看板每天收到新观测值，需要刷新预测，而不是手工重建整个流程。

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(2024)
n = 260
dates = pd.date_range("2023-01-01", periods=n, freq="D")
dow = dates.dayofweek.to_numpy()
campaign = rng.binomial(1, 0.10, n)
incidents = rng.poisson(0.15, n)
tickets = 900 + 130 * (dow < 5) + 220 * campaign + 80 * incidents + 40 * np.sin(np.linspace(0, 8*np.pi, n)) + rng.normal(0, 45, n)
ops = pd.DataFrame({"date": dates, "tickets": np.maximum(tickets, 1), "campaign": campaign, "incidents": incidents})

initial = ops.iloc[:200].copy()
new_batch = ops.iloc[200:230].copy()
holdout = ops.iloc[230:].copy()

In [ ]:
from PipelineTS.pipeline import ModelPipeline

pipe = ModelPipeline(
    time_col="date",
    target_col="tickets",
    lags=14,
    known_covariates=["campaign"],
    past_covariates=["incidents"],
    include_models=["random_forest", "multi_output_model"],
    quantile=0.9,
    cv=2,
    random_forest__n_estimators=120,
)
pipe.fit(initial)
before_update = pipe.predict(14, future_covariates=holdout[["date", "campaign"]].head(14))
before_update.head()

In [ ]:
pipe.update(new_batch, refit_all=False)
after_update = pipe.predict(14, future_covariates=holdout[["date", "campaign"]].head(14))
after_update.head()

In [ ]:
from PipelineTS.pipeline import SmartRouter

router = SmartRouter(
    time_col="date",
    target_col="tickets",
    known_covariates=["campaign"],
    past_covariates=["incidents"],
    preset="fast",
    include_models=["random_forest", "multi_output_model", "extra_forest"],
    quantile=0.9,
    time_limit=60,
)
router.fit(initial)
router.update(new_batch, refit_all=False)
router.predict(14, future_covariates=holdout[["date", "campaign"]].head(14)).head()

In [ ]:
from pathlib import Path

path = Path("../tmp_ops_router.pts")
router.save(path, metadata={"scenario": "operations_tickets", "refresh": "daily"})
loaded_router = SmartRouter.load(path)
loaded_router.predict(7, future_covariates=holdout[["date", "campaign"]].head(7)).head()

In [ ]:
router.plot(n=14, history_tail=80, lang="zh")
router.plot_leaderboard(lang="zh")